In [1]:
import pandas as pd
import json
from sqlalchemy import create_engine

In [2]:
import pandas as pd
import json
import time


def load_json_file(file_path, file_name, current, total):
    print(f"\n[{current}/{total}] Starting: {file_name}")
    
    start_time = time.time()

    with open(file_path, 'r') as f:
        data = [json.loads(line) for line in f]

    df = pd.DataFrame(data)

    end_time = time.time()
    elapsed = end_time - start_time

    progress = (current / total) * 100
    remaining = total - current

    print(f"✅ Completed: {file_name}")
    print(f"📊 Rows loaded: {len(df):,}")
    print(f"📈 Overall progress: {progress:.0f}%")
    print(f"⏱️ Time taken: {elapsed:.2f} seconds")
    print(f"📌 Files remaining: {remaining}")

    return df


# Total number of files
total_files = 5


businesses_df = load_json_file(
    '../data/yelp_academic_dataset_business.json',
    'business',
    1,
    total_files
)

checkin_df = load_json_file(
    '../data/yelp_academic_dataset_checkin.json',
    'checkin',
    2,
    total_files
)

review_df = load_json_file(
    '../data/yelp_academic_dataset_review.json',
    'review',
    3,
    total_files
)

tip_df = load_json_file(
    '../data/yelp_academic_dataset_tip.json',
    'tip',
    4,
    total_files
)

user_df = load_json_file(
    '../data/yelp_academic_dataset_user.json',
    'user',
    5,
    total_files
)

print("\n🎉 All JSON files loaded successfully!")


[1/5] Starting: business
✅ Completed: business
📊 Rows loaded: 150,346
📈 Overall progress: 20%
⏱️ Time taken: 0.72 seconds
📌 Files remaining: 4

[2/5] Starting: checkin
✅ Completed: checkin
📊 Rows loaded: 131,930
📈 Overall progress: 40%
⏱️ Time taken: 0.38 seconds
📌 Files remaining: 3

[3/5] Starting: review
✅ Completed: review
📊 Rows loaded: 6,990,280
📈 Overall progress: 60%
⏱️ Time taken: 440.84 seconds
📌 Files remaining: 2

[4/5] Starting: tip
✅ Completed: tip
📊 Rows loaded: 908,915
📈 Overall progress: 80%
⏱️ Time taken: 3.80 seconds
📌 Files remaining: 1

[5/5] Starting: user
✅ Completed: user
📊 Rows loaded: 1,987,897
📈 Overall progress: 100%
⏱️ Time taken: 287.27 seconds
📌 Files remaining: 0

🎉 All JSON files loaded successfully!


In [3]:
print(businesses_df.shape)
print(checkin_df.shape)
print(review_df.shape)
print(tip_df.shape)
print(user_df.shape)


(150346, 14)
(131930, 2)
(6990280, 9)
(908915, 5)
(1987897, 22)


In [4]:
businesses_df.drop(['attributes','hours'],axis = 1, inplace = True)

In [5]:


engine = create_engine('sqlite:///../data/yelp.db')


def load_dataframe(df, table_name, engine, current, total, chunk_size=10000):
    print(f"\n[{current}/{total}] Starting: {table_name}")
    print(f"📊 Total rows: {len(df):,}")

    start_time = time.time()

    # Create/replace table using the first chunk
    first_chunk = True

    for start in range(0, len(df), chunk_size):

        chunk = df.iloc[start:start + chunk_size]

        if first_chunk:
            chunk.to_sql(
                table_name,
                con=engine,
                if_exists='replace',
                index=False
            )
            first_chunk = False
        else:
            chunk.to_sql(
                table_name,
                con=engine,
                if_exists='append',
                index=False
            )

        # Progress
        rows_loaded = min(start + chunk_size, len(df))
        progress = (rows_loaded / len(df)) * 100

        print(
            f"\r⏳ {table_name}: "
            f"{rows_loaded:,}/{len(df):,} rows "
            f"({progress:.1f}%)",
            end="",
            flush=True
        )

    elapsed = time.time() - start_time

    print(f"\n✅ Completed: {table_name}")
    print(f"⏱️ Time taken: {elapsed:.2f} seconds")
    print(f"📌 Tables remaining: {total - current}")


# Total number of tables
total_tables = 5


load_dataframe(
    businesses_df,
    'business',
    engine,
    1,
    total_tables
)

load_dataframe(
    review_df,
    'review',
    engine,
    2,
    total_tables
)

load_dataframe(
    user_df,
    'user',
    engine,
    3,
    total_tables
)

load_dataframe(
    tip_df,
    'tip',
    engine,
    4,
    total_tables
)

load_dataframe(
    checkin_df,
    'checkin',
    engine,
    5,
    total_tables
)

print("\n🎉 All tables loaded successfully!")


[1/5] Starting: business
📊 Total rows: 150,346
⏳ business: 150,346/150,346 rows (100.0%)
✅ Completed: business
⏱️ Time taken: 1.02 seconds
📌 Tables remaining: 4

[2/5] Starting: review
📊 Total rows: 6,990,280
⏳ review: 6,990,280/6,990,280 rows (100.0%)
✅ Completed: review
⏱️ Time taken: 123.38 seconds
📌 Tables remaining: 3

[3/5] Starting: user
📊 Total rows: 1,987,897
⏳ user: 1,987,897/1,987,897 rows (100.0%)
✅ Completed: user
⏱️ Time taken: 75.52 seconds
📌 Tables remaining: 2

[4/5] Starting: tip
📊 Total rows: 908,915
⏳ tip: 908,915/908,915 rows (100.0%)
✅ Completed: tip
⏱️ Time taken: 6.33 seconds
📌 Tables remaining: 1

[5/5] Starting: checkin
📊 Total rows: 131,930
⏳ checkin: 131,930/131,930 rows (100.0%)
✅ Completed: checkin
⏱️ Time taken: 2.90 seconds
📌 Tables remaining: 0

🎉 All tables loaded successfully!
